
# Gold Trend Engine V7 — Reconstruction & Crypto-Daily Validation Harness

**Purpose.** The source script (`Gold Trend Engine V7 - Structure Matrix`) is closed-source /
protected on TradingView. Nothing in this notebook is the original Pine code — it is a
**best-effort reconstruction** of the *documented* architecture (HMA trend base, DI/ADX
strength, multi-speed momentum, ATR structure/protection, Kaufman-style efficiency quality
filter, weighted Power/Confidence gate with a Normal-Reversal path and a looser
Structural-Override path), built from the public description + the input panel you shared.

**What this is for:** deciding whether the *methodology* — not the exact proprietary formula —
has a real edge on crypto daily swing, and whether the parameter set discussed (protection ATR,
reversal threshold, HMA lengths, quality thresholds, override power) survives out-of-sample
testing instead of just "looking right" on a chart.

**What this is NOT:** a guarantee that this notebook's Power/Confidence weighting matches the
real indicator bar-for-bar. Treat the internal weights (Section 4) as an explicit, editable
assumption — not ground truth. Validate the *shape* of the results (does tightening protection
ATR reduce whipsaw, does the override path fire too often on crypto, etc.), not the exact P&L
number.

**Data.** Attempts live OHLCV via `ccxt` (Binance spot, daily bars). If the exchange is
unreachable from wherever this notebook runs (e.g. sandboxed/no-internet environments), it
falls back to a seeded synthetic GBM+jump generator per token so the harness is always
end-to-end runnable — and it tells you which source was actually used, per token, in the
summary table. **Do not trust results computed on synthetic fallback data as real backtest
output** — re-run with `ccxt` reachable before drawing conclusions.


## 1. Setup & config

All tunable parameters live here — this is the single place to edit for a re-run.

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

try:
    import ccxt
    HAVE_CCXT = True
except ImportError:
    HAVE_CCXT = False

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.random.seed(42)


In [ ]:

# ============================================================================
# TOP-10 CRYPTO UNIVERSE (excl. stablecoins) — EDIT THIS to match current
# market-cap rankings at run time; rankings drift and this list will go stale.
# Symbols are Binance spot pairs vs USDT.
# ============================================================================
TOP10_SYMBOLS = [
    "BTC/USDT", "ETH/USDT", "BNB/USDT", "SOL/USDT", "XRP/USDT",
    "DOGE/USDT", "ADA/USDT", "TRX/USDT", "AVAX/USDT", "LINK/USDT",
]

TIMEFRAME = "1d"
LOOKBACK_BARS = 1500          # ~4 years of daily bars, adjust to token listing age
FEE_BPS = 10                  # 0.10% taker, per side (Binance spot default tier)
SLIPPAGE_BPS = 5               # additional modeled slippage per side

# ----------------------------------------------------------------------------
# ENGINE PARAMETERS — the exact fields from the indicator's input panel.
# Two configs: GOLD_DEFAULT (as shipped) vs CRYPTO_CANDIDATE (the retune we
# discussed). The walk-forward section (9) will search around CRYPTO_CANDIDATE.
# ----------------------------------------------------------------------------
GOLD_DEFAULT = dict(
    fast_hma=8, slow_hma=21, atr_len=14, di_len=7, adx_smooth=6,
    mom_fast=2, mom_mid=5, mom_slow=9,
    struct_lookback=5, protection_atr=1.35, min_reversal_atr=0.6,
    eff_len=10, soft_eff=0.18, soft_adx=15,
    init_power=20, init_conf=4,
    normal_power=24, normal_conf=4,
    override_power=12, override_conf=4,
    min_bars_reversal=2,
)

CRYPTO_CANDIDATE = dict(
    fast_hma=11, slow_hma=40, atr_len=14, di_len=14, adx_smooth=11,
    mom_fast=3, mom_mid=7, mom_slow=14,
    struct_lookback=8, protection_atr=2.0, min_reversal_atr=1.2,
    eff_len=14, soft_eff=0.25, soft_adx=21,
    init_power=20, init_conf=4,
    normal_power=28, normal_conf=4,
    override_power=15, override_conf=4,
    min_bars_reversal=4,
)


## 2. Data acquisition — live Binance via `ccxt`, seeded synthetic fallback

In [ ]:

def fetch_ohlcv_binance(symbol, timeframe="1d", limit=1500):
    ex = ccxt.binance({"enableRateLimit": True})
    all_rows = []
    since = ex.milliseconds() - limit * 24 * 60 * 60 * 1000
    while True:
        batch = ex.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=1000)
        if not batch:
            break
        all_rows += batch
        since = batch[-1][0] + 1
        if len(batch) < 1000:
            break
        if len(all_rows) >= limit:
            break
    df = pd.DataFrame(all_rows, columns=["ts", "open", "high", "low", "close", "volume"])
    df["date"] = pd.to_datetime(df["ts"], unit="ms")
    df = df.set_index("date").drop(columns="ts")
    return df.tail(limit)


def fetch_synthetic(symbol, n_bars=1500, seed=None):
    '''Seeded GBM + occasional jump/vol-cluster generator, purely for making
    this notebook runnable with zero network access. NOT a substitute for real
    data — see the data-source column in the summary table.'''
    rng = np.random.default_rng(seed if seed is not None else abs(hash(symbol)) % (2**32))
    n = n_bars
    mu, sigma = 0.0004, 0.035          # crude crypto-like daily drift/vol
    vol_regime = 1 + 0.6 * np.abs(np.sin(np.linspace(0, 6 * np.pi, n)))
    rets = rng.normal(mu, sigma, n) * vol_regime
    jump_mask = rng.random(n) < 0.02
    rets[jump_mask] += rng.normal(0, 0.08, jump_mask.sum())
    price = 100 * np.exp(np.cumsum(rets))
    high = price * (1 + np.abs(rng.normal(0, 0.012, n)))
    low = price * (1 - np.abs(rng.normal(0, 0.012, n)))
    openp = np.roll(price, 1)
    openp[0] = price[0]
    vol = rng.lognormal(10, 1, n)
    idx = pd.date_range(end=pd.Timestamp.today().normalize(), periods=n, freq="D")
    df = pd.DataFrame({"open": openp, "high": high, "low": low, "close": price,
                        "volume": vol}, index=idx)
    return df


def get_data(symbol, limit=LOOKBACK_BARS):
    if HAVE_CCXT:
        try:
            df = fetch_ohlcv_binance(symbol, TIMEFRAME, limit)
            if len(df) > 200:
                return df, "live:binance"
        except Exception as e:
            print(f"  [{symbol}] live fetch failed ({type(e).__name__}: {e}) -> synthetic fallback")
    return fetch_synthetic(symbol, limit), "synthetic:fallback"


## 3. Indicator library (HMA, ATR, DI/ADX, efficiency ratio, momentum)

In [ ]:

def wma(s, n):
    w = np.arange(1, n + 1)
    return s.rolling(n).apply(lambda x: np.dot(x, w) / w.sum(), raw=True)

def hma(s, n):
    half = wma(s, n // 2)
    full = wma(s, n)
    raw = 2 * half - full
    return wma(raw, int(round(np.sqrt(n))))

def atr(df, n):
    hl = df["high"] - df["low"]
    hc = (df["high"] - df["close"].shift()).abs()
    lc = (df["low"] - df["close"].shift()).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()

def di_adx(df, di_len, adx_smooth):
    up = df["high"].diff()
    down = -df["low"].diff()
    plus_dm = np.where((up > down) & (up > 0), up, 0.0)
    minus_dm = np.where((down > up) & (down > 0), down, 0.0)
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - df["close"].shift()).abs(),
        (df["low"] - df["close"].shift()).abs(),
    ], axis=1).max(axis=1)
    atr_di = tr.ewm(alpha=1 / di_len, adjust=False).mean()
    plus_di = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1 / di_len, adjust=False).mean() / atr_di
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1 / di_len, adjust=False).mean() / atr_di
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    adx = dx.ewm(alpha=1 / adx_smooth, adjust=False).mean()
    return plus_di, minus_di, adx

def efficiency_ratio(s, n):
    change = (s - s.shift(n)).abs()
    vol = s.diff().abs().rolling(n).sum()
    return (change / vol.replace(0, np.nan)).fillna(0)

def roc(s, n):
    return (s / s.shift(n) - 1) * 100



## 4. Trend Engine reconstruction

**Explicit assumption (edit `power_weights` / `conf_rules` below to test alternatives):**
Power is a 0-100-ish weighted sum of five agreement components (trend slope, DI dominance,
HMA fast/slow relationship, momentum-stack agreement, structural alignment). Confidence is a
simple count (0-6) of how many of the six listed conditions agree with the candidate direction.
A **Normal Reversal** requires a structural break AND Power ≥ `normal_power` AND
Confidence ≥ `normal_conf` AND bars-since-last-flip ≥ `min_bars_reversal`. A **Structural
Override** requires a stronger structural break AND Power ≥ `override_power` AND
Confidence ≥ `override_conf`, bypassing the min-bars cooldown — this is the path flagged
earlier as the most likely whipsaw backdoor on crypto.


In [ ]:

def compute_engine(df, cfg):
    d = df.copy()
    d["hma_fast"] = hma(d["close"], cfg["fast_hma"])
    d["hma_slow"] = hma(d["close"], cfg["slow_hma"])
    d["atr"] = atr(d, cfg["atr_len"])
    d["plus_di"], d["minus_di"], d["adx"] = di_adx(d, cfg["di_len"], cfg["adx_smooth"])

    d["mom_fast"] = roc(d["close"], cfg["mom_fast"])
    d["mom_mid"] = roc(d["close"], cfg["mom_mid"])
    d["mom_slow"] = roc(d["close"], cfg["mom_slow"])

    d["swing_high"] = d["high"].rolling(cfg["struct_lookback"]).max()
    d["swing_low"] = d["low"].rolling(cfg["struct_lookback"]).min()

    d["efficiency"] = efficiency_ratio(d["close"], cfg["eff_len"])
    d["slope"] = d["hma_fast"].diff()

    return d


def power_confidence(d, i, direction, cfg):
    '''direction: +1 bullish candidate, -1 bearish candidate.
    Returns (power 0-100, confidence 0-6).'''
    row = d.iloc[i]
    checks = []
    # 1. HMA fast/slow relationship
    checks.append(np.sign(row["hma_fast"] - row["hma_slow"]) == direction)
    # 2. HMA slope
    checks.append(np.sign(row["slope"]) == direction)
    # 3. DI dominance
    checks.append(np.sign(row["plus_di"] - row["minus_di"]) == direction)
    # 4. ADX trend-quality gate (direction-agnostic strength check)
    checks.append(row["adx"] >= cfg["soft_adx"])
    # 5. Momentum stack agreement (majority of 3 speeds)
    mom_signs = np.sign([row["mom_fast"], row["mom_mid"], row["mom_slow"]])
    checks.append(np.sign(mom_signs.sum()) == direction)
    # 6. Efficiency / quality
    checks.append(row["efficiency"] >= cfg["soft_eff"])

    confidence = int(sum(checks))
    weights = np.array([20, 15, 20, 15, 20, 10])  # sums to 100; edit to test sensitivity
    power = float(np.dot(weights, checks))
    return power, confidence


def run_state_machine(d, cfg):
    n = len(d)
    state = np.zeros(n, dtype=int)          # +1 bull, -1 bear, 0 flat/undetermined
    protection = np.full(n, np.nan)
    extreme = np.full(n, np.nan)
    events = []                              # (index, 'BULL'/'BEAR', mode)
    bars_since_flip = 10**6
    cur_state = 0

    warmup = max(cfg["slow_hma"], cfg["struct_lookback"], cfg["eff_len"], cfg["di_len"]) + 5

    for i in range(warmup, n):
        row = d.iloc[i]
        if np.isnan(row["hma_slow"]) or np.isnan(row["atr"]):
            state[i] = cur_state
            continue

        if cur_state == 0:
            # initial establishment
            for direction, label in [(1, "BULL"), (-1, "BEAR")]:
                p, c = power_confidence(d, i, direction, cfg)
                if p >= cfg["init_power"] and c >= cfg["init_conf"]:
                    cur_state = direction
                    extreme[i] = row["close"]
                    protection[i] = row["close"] - direction * cfg["protection_atr"] * row["atr"]
                    bars_since_flip = 0
                    events.append((i, label, "INIT"))
                    break
            state[i] = cur_state
            continue

        # maintain trailing extreme + protection level while in a state
        prev_extreme = extreme[i - 1] if not np.isnan(extreme[i - 1]) else row["close"]
        if cur_state == 1:
            new_extreme = max(prev_extreme, row["high"])
        else:
            new_extreme = min(prev_extreme, row["low"])
        extreme[i] = new_extreme
        protection[i] = new_extreme - cur_state * cfg["protection_atr"] * row["atr"]

        # structural break test: close beyond swing high/low against current state
        struct_break = (row["close"] < row["swing_low"]) if cur_state == 1 else (row["close"] > row["swing_high"])
        reversal_move = abs(row["close"] - prev_extreme)
        min_move_ok = reversal_move >= cfg["min_reversal_atr"] * row["atr"]

        candidate_dir = -cur_state
        p, c = power_confidence(d, i, candidate_dir, cfg)

        flipped = False
        if struct_break and min_move_ok:
            if p >= cfg["override_power"] and c >= cfg["override_conf"]:
                cur_state = candidate_dir
                events.append((i, "BULL" if cur_state == 1 else "BEAR", "OVERRIDE"))
                flipped = True
            elif (bars_since_flip >= cfg["min_bars_reversal"]
                  and p >= cfg["normal_power"] and c >= cfg["normal_conf"]):
                cur_state = candidate_dir
                events.append((i, "BULL" if cur_state == 1 else "BEAR", "NORMAL"))
                flipped = True

        if flipped:
            extreme[i] = row["close"]
            protection[i] = row["close"] - cur_state * cfg["protection_atr"] * row["atr"]
            bars_since_flip = 0
        else:
            bars_since_flip += 1

        state[i] = cur_state

    d = d.copy()
    d["state"] = state
    d["protection"] = protection
    d["extreme"] = extreme
    ev = pd.DataFrame(events, columns=["idx", "label", "mode"])
    ev["date"] = d.index[ev["idx"]] if len(ev) else []
    return d, ev


## 5. Trade simulation (long/short swing, fee + slippage modeled)

In [ ]:

def simulate_trades(d, ev, fee_bps=FEE_BPS, slippage_bps=SLIPPAGE_BPS):
    cost = (fee_bps + slippage_bps) / 10000
    trades = []
    open_trade = None

    for _, row in ev.iterrows():
        i = row["idx"]
        direction = 1 if row["label"] == "BULL" else -1
        entry_i = min(i + 1, len(d) - 1)          # act on next bar open
        entry_price = d["open"].iloc[entry_i] * (1 + cost * direction)
        entry_date = d.index[entry_i]
        init_protection = d["protection"].iloc[i]
        risk_per_unit = abs(entry_price - init_protection)

        if open_trade is not None:
            exit_price = d["open"].iloc[entry_i] * (1 - cost * open_trade["direction"])
            pnl = (exit_price - open_trade["entry_price"]) * open_trade["direction"]
            r_mult = pnl / open_trade["risk_per_unit"] if open_trade["risk_per_unit"] > 0 else np.nan
            trades.append({**open_trade, "exit_price": exit_price, "exit_date": entry_date,
                           "pnl": pnl, "r_multiple": r_mult,
                           "bars_held": entry_i - open_trade["entry_idx"]})

        open_trade = dict(direction=direction, entry_price=entry_price, entry_date=entry_date,
                           entry_idx=entry_i, risk_per_unit=risk_per_unit, mode=row["mode"])

    if open_trade is not None:
        exit_price = d["close"].iloc[-1] * (1 - cost * open_trade["direction"])
        pnl = (exit_price - open_trade["entry_price"]) * open_trade["direction"]
        r_mult = pnl / open_trade["risk_per_unit"] if open_trade["risk_per_unit"] > 0 else np.nan
        trades.append({**open_trade, "exit_price": exit_price, "exit_date": d.index[-1],
                       "pnl": pnl, "r_multiple": r_mult,
                       "bars_held": len(d) - 1 - open_trade["entry_idx"]})

    return pd.DataFrame(trades)


## 6. Performance metrics

In [ ]:

def trade_metrics(trades, d):
    if trades is None or len(trades) == 0:
        return dict(n_trades=0)
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    win_rate = len(wins) / len(trades)
    avg_r = trades["r_multiple"].mean()
    profit_factor = (wins["pnl"].sum() / abs(losses["pnl"].sum())) if len(losses) and losses["pnl"].sum() != 0 else np.nan
    expectancy_r = avg_r

    # equity curve in R, one unit risk per trade, no compounding (conservative)
    eq = trades["r_multiple"].cumsum()
    running_max = eq.cummax()
    dd = eq - running_max
    max_dd_r = dd.min()

    years = (d.index[-1] - d.index[0]).days / 365.25
    trades_per_year = len(trades) / years if years > 0 else np.nan

    daily_r = trades.set_index("exit_date")["r_multiple"]
    sharpe_like = (daily_r.mean() / daily_r.std() * np.sqrt(trades_per_year)) if daily_r.std() > 0 else np.nan

    return dict(
        n_trades=len(trades), win_rate=win_rate, avg_r=avg_r,
        profit_factor=profit_factor, expectancy_r=expectancy_r,
        max_dd_r=max_dd_r, avg_bars_held=trades["bars_held"].mean(),
        trades_per_year=trades_per_year, sharpe_like=sharpe_like,
        pct_override_trades=(trades["mode"] == "OVERRIDE").mean(),
    )


## 7. Run across the top-10 universe (GOLD_DEFAULT vs CRYPTO_CANDIDATE)

In [ ]:

def run_universe(symbols, cfg, limit=LOOKBACK_BARS, label=""):
    results = {}
    for sym in symbols:
        print(f"[{label}] {sym} ...", end=" ")
        df, source = get_data(sym, limit)
        d = compute_engine(df, cfg)
        d, ev = run_state_machine(d, cfg)
        trades = simulate_trades(d, ev)
        m = trade_metrics(trades, d)
        m["symbol"] = sym
        m["data_source"] = source
        results[sym] = dict(df=d, events=ev, trades=trades, metrics=m)
        print(f"{m.get('n_trades', 0)} trades, source={source}")
    return results

print("=== GOLD_DEFAULT params on crypto daily ===")
results_gold_default = run_universe(TOP10_SYMBOLS, GOLD_DEFAULT, label="gold-default")

print()
print("=== CRYPTO_CANDIDATE params on crypto daily ===")
results_crypto = run_universe(TOP10_SYMBOLS, CRYPTO_CANDIDATE, label="crypto-candidate")


In [ ]:

def summary_table(results):
    rows = [r["metrics"] for r in results.values()]
    return pd.DataFrame(rows).set_index("symbol")[
        ["n_trades", "win_rate", "avg_r", "profit_factor", "expectancy_r",
         "max_dd_r", "avg_bars_held", "trades_per_year", "sharpe_like",
         "pct_override_trades", "data_source"]
    ]

print("GOLD_DEFAULT (as-shipped, gold-tuned) on crypto daily:")
display(summary_table(results_gold_default))

print("\nCRYPTO_CANDIDATE (retuned) on crypto daily:")
display(summary_table(results_crypto))



## 8. Portfolio-level view & correlation check

Ten tokens is **not** ten independent trials. Crypto majors/alts are heavily beta-correlated
to BTC, so a "win" replicated across all 10 is closer to n=1-2 effective degrees of freedom
(BTC regime + maybe one alt-season factor) than a robust n=10 confirmation. This section
builds an equal-weight portfolio equity curve and a correlation matrix of per-token equity
curves specifically to surface that.


In [ ]:

def build_equity_curve(trades, index):
    if trades is None or len(trades) == 0:
        return pd.Series(0.0, index=index)
    r = trades.set_index("exit_date")["r_multiple"]
    r = r.reindex(index, fill_value=0.0)
    return r.cumsum()

def portfolio_view(results, title):
    curves = {}
    for sym, r in results.items():
        idx = r["df"].index
        curves[sym] = build_equity_curve(r["trades"], idx)
    curve_df = pd.DataFrame(curves).ffill().fillna(0)
    portfolio_eq = curve_df.mean(axis=1)  # equal-weight, in R-units

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    curve_df.plot(ax=axes[0], legend=False, alpha=0.4)
    portfolio_eq.plot(ax=axes[0], color="black", linewidth=2.5, label="Equal-weight portfolio")
    axes[0].set_title(f"{title}: per-token equity (R) + portfolio")
    axes[0].legend()

    corr = curve_df.diff().corr()
    im = axes[1].imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    axes[1].set_xticks(range(len(corr))); axes[1].set_xticklabels(corr.columns, rotation=90)
    axes[1].set_yticks(range(len(corr))); axes[1].set_yticklabels(corr.columns)
    axes[1].set_title("Correlation of daily equity changes")
    fig.colorbar(im, ax=axes[1], fraction=0.046)
    plt.tight_layout()
    plt.show()

    avg_corr = corr.values[np.triu_indices_from(corr.values, k=1)].mean()
    print(f"Average pairwise correlation of per-token strategy equity changes: {avg_corr:.2f}")
    print("(High values here mean your '10-token validation' is mostly re-testing the same BTC-beta bet.)")
    return curve_df, portfolio_eq

_ = portfolio_view(results_crypto, "CRYPTO_CANDIDATE")



## 9. Walk-forward parameter validation (the actual point of this notebook)

Split each token's history into an **in-sample (train)** window and an **out-of-sample
(test)** window. Grid-search the leverage-point parameters on train only, pick the best
config by expectancy (with a minimum-trade-count filter so a 2-trade lucky config can't win),
then freeze it and run once on test. Large in-sample vs out-of-sample degradation = overfit,
not edge.


In [ ]:

import itertools

GRID = dict(
    protection_atr=[1.5, 1.8, 2.0, 2.4],
    min_reversal_atr=[0.8, 1.0, 1.2, 1.5],
    slow_hma=[21, 34, 40, 55],
    soft_adx=[15, 18, 21, 25],
    override_power=[12, 15, 18],
)

TRAIN_FRAC = 0.7
MIN_TRADES = 8

def split_train_test(df, frac=TRAIN_FRAC):
    cut = int(len(df) * frac)
    return df.iloc[:cut], df.iloc[cut:]

def evaluate_config(df_slice, cfg):
    d = compute_engine(df_slice, cfg)
    d, ev = run_state_machine(d, cfg)
    trades = simulate_trades(d, ev)
    return trade_metrics(trades, d), trades

def walk_forward_one(symbol, base_cfg, grid, limit=LOOKBACK_BARS):
    df, source = get_data(symbol, limit)
    train, test = split_train_test(df)

    keys = list(grid.keys())
    best_cfg, best_score, best_train_metrics = None, -np.inf, None
    for combo in itertools.product(*grid.values()):
        cfg = {**base_cfg, **dict(zip(keys, combo))}
        m, _ = evaluate_config(train, cfg)
        if m.get("n_trades", 0) < MIN_TRADES:
            continue
        score = m.get("expectancy_r", -np.inf)
        if score is not None and score > best_score:
            best_score, best_cfg, best_train_metrics = score, cfg, m

    if best_cfg is None:
        return dict(symbol=symbol, data_source=source, status="no config met MIN_TRADES on train")

    test_metrics, _ = evaluate_config(test, best_cfg)
    return dict(
        symbol=symbol, data_source=source, status="ok",
        best_cfg={k: best_cfg[k] for k in keys},
        train_expectancy_r=best_train_metrics.get("expectancy_r"),
        train_n=best_train_metrics.get("n_trades"),
        test_expectancy_r=test_metrics.get("expectancy_r"),
        test_n=test_metrics.get("n_trades"),
        test_win_rate=test_metrics.get("win_rate"),
        test_max_dd_r=test_metrics.get("max_dd_r"),
    )

# WARNING: full grid x 10 symbols is expensive (4*4*4*4*3 = 768 backtests/token).
# Trim GRID above for a quick pass, or subsample SYMBOLS_FOR_WF below.
SYMBOLS_FOR_WF = TOP10_SYMBOLS[:3]   # widen once you've confirmed runtime is acceptable

wf_results = [walk_forward_one(sym, CRYPTO_CANDIDATE, GRID) for sym in SYMBOLS_FOR_WF]
wf_df = pd.DataFrame(wf_results)
display(wf_df)



## 10. Sanity-check chart (state overlay on one token)

Visual gut-check for one symbol: does the state flip roughly where you'd expect, and how
often does the OVERRIDE path fire vs NORMAL?


In [ ]:

def plot_state(results_dict, symbol):
    d = results_dict[symbol]["df"]
    ev = results_dict[symbol]["events"]

    fig, ax = plt.subplots(figsize=(15, 6))
    ax.plot(d.index, d["close"], color="black", linewidth=0.8, label="Close")
    ax.plot(d.index, d["protection"], color="orange", linewidth=0.8, alpha=0.7, label="Protection level")

    bulls = ev[ev["label"] == "BULL"]
    bears = ev[ev["label"] == "BEAR"]
    ax.scatter(d.index[bulls["idx"]], d["close"].iloc[bulls["idx"]], marker="^", color="green", s=60, zorder=5, label="BULL flip")
    ax.scatter(d.index[bears["idx"]], d["close"].iloc[bears["idx"]], marker="v", color="red", s=60, zorder=5, label="BEAR flip")

    override = ev[ev["mode"] == "OVERRIDE"]
    ax.scatter(d.index[override["idx"]], d["close"].iloc[override["idx"]], marker="o",
               facecolors="none", edgecolors="purple", s=120, linewidths=1.5, zorder=6, label="via OVERRIDE")

    ax.set_title(f"{symbol} — state transitions ({(ev['mode']=='OVERRIDE').sum()} override / {len(ev)} total)")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

plot_state(results_crypto, TOP10_SYMBOLS[0])



## 11. Limitations & next steps

- **Formula reconstruction, not the real engine.** `power_confidence()` weights and the
  6-condition list are documented assumptions. Re-run Section 9 sensitivity with different
  `weights` arrays before trusting any single number.
- **No fees for a `STRUCTURAL_OVERRIDE` misfire cascade** beyond the modeled per-trade cost —
  if `pct_override_trades` is high, that's the whipsaw-backdoor risk flagged earlier;
  investigate those trades specifically (`trades[trades.mode=="OVERRIDE"]`).
- **Survivorship in `TOP10_SYMBOLS`.** Today's top 10 weren't necessarily top 10 for the whole
  `LOOKBACK_BARS` window — this doesn't correct for that. For a stricter test, either shorten
  the lookback to the youngest listing's full history or accept this as a known bias.
- **No funding-rate modeling.** If this becomes a perp strategy rather than spot swing, add
  funding P&L — daily funding drag/carry meaningfully changes expectancy on multi-day holds.
- **Correlation, not independence** (Section 8) — treat the 10-token result as roughly 2-3
  effective independent tests, not 10.
- **Next step if CRYPTO_CANDIDATE survives walk-forward:** widen `SYMBOLS_FOR_WF` to all 10,
  widen `GRID`, and re-run with `LOOKBACK_BARS` covering at least 2 full crypto cycles
  (bull + bear) before sizing anything real against it.
